In [ ]:
# Unicode Scan and Fix

This notebook scans the workspace for malformed Unicode and invalid UTF-16 surrogate pairs, identifies affected files and lines, replaces invalid characters safely, enforces UTF-8 encoding, and validates the fix.


# Unicode Scan and Fix

This notebook scans the workspace for malformed Unicode and invalid UTF-16 surrogate pairs, identifies affected files and lines, replaces invalid characters safely, enforces UTF-8 encoding, and validates the fix.

In [ ]:
import os
import re
from pathlib import Path

try:
    import chardet
except ImportError:
    chardet = None

root = Path(r'c:\Users\Kate\Desktop\Projects\roadmap')
text_extensions = {
    '.js', '.jsx', '.ts', '.tsx', '.json', '.md', '.markdown', '.css', '.scss', '.html', '.htm', '.csv', '.txt', '.yml', '.yaml', '.xml', '.env', '.ini', '.json5', '.ipynb', '.jsx', '.tsx'
}
skip_extensions = {
    '.png', '.jpg', '.jpeg', '.gif', '.ico', '.bmp', '.exe', '.dll', '.wasm', '.bin', '.pdf', '.zip', '.tar', '.gz', '.7z', '.mp3', '.mp4', '.avi', '.mov', '.mpeg', '.ttf', '.woff', '.woff2', '.eot', '.otf', '.map', '.lock'
}

print(f'Workspace root: {root}')
print('chardet available:' , chardet is not None)

Workspace root: c:\Users\Kate\Desktop\Projects\roadmap
chardet available: False


## Scan Workspace for Invalid Unicode

Recursively scan all likely text files and detect invalid UTF-8 sequences or unpaired UTF-16 surrogate code points.

In [8]:
invalid_files = []

ignore_dirs = {'node_modules', '.venv', 'dist', '__pycache__', '.git'}

for path in root.rglob('*'):
    if not path.is_file():
        continue
    if any(part in ignore_dirs for part in path.parts):
        continue
    if path.suffix.lower() in skip_extensions:
        continue
    is_text = path.suffix.lower() in text_extensions or path.name in {'package.json', '.gitignore', 'README.md', 'LICENSE', 'vite.config.js', 'eslint.config.js'}
    if not is_text and path.stat().st_size > 1024 * 1024:
        continue

    data = path.read_bytes()
    try:
        text = data.decode('utf-8')
    except Exception as decode_error:
        try:
            text = data.decode('utf-8', errors='surrogatepass')
        except Exception:
            invalid_files.append((path, 'utf8-decode-fail', str(decode_error)))
            continue
        surrogates = [(i, hex(ord(ch)), text.count('\n', 0, i) + 1) for i, ch in enumerate(text) if 0xD800 <= ord(ch) <= 0xDFFF]
        invalid_files.append((path, 'surrogates', surrogates[:20]))
        continue

    surrogates = [(i, hex(ord(ch)), text.count('\n', 0, i) + 1) for i, ch in enumerate(text) if 0xD800 <= ord(ch) <= 0xDFFF]
    if surrogates:
        invalid_files.append((path, 'surrogates', surrogates[:20]))

invalid_files

[]

In [3]:
print('Invalid file count:', len(invalid_files))
for path, kind, detail in invalid_files[:50]:
    print(path.relative_to(root), kind, detail if kind == 'utf8-decode-fail' else f'{len(detail)} surrogate locations')


Invalid file count: 411
.venv\Lib\site-packages\pip\__pycache__\__init__.cpython-313.pyc utf8-decode-fail 'utf-8' codec can't decode byte 0xf3 in position 0: invalid continuation byte
.venv\Lib\site-packages\pip\__pycache__\__main__.cpython-313.pyc utf8-decode-fail 'utf-8' codec can't decode byte 0xf3 in position 0: invalid continuation byte
.venv\Lib\site-packages\pip\__pycache__\__pip-runner__.cpython-313.pyc utf8-decode-fail 'utf-8' codec can't decode byte 0xf3 in position 0: invalid continuation byte
.venv\Lib\site-packages\pip\_vendor\__pycache__\typing_extensions.cpython-313.pyc utf8-decode-fail 'utf-8' codec can't decode byte 0xf3 in position 0: invalid continuation byte
.venv\Lib\site-packages\pip\_vendor\__pycache__\__init__.cpython-313.pyc utf8-decode-fail 'utf-8' codec can't decode byte 0xf3 in position 0: invalid continuation byte
.venv\Lib\site-packages\pip\_vendor\urllib3\__pycache__\connection.cpython-313.pyc utf8-decode-fail 'utf-8' codec can't decode byte 0xf3 in posit

## Identify Affected Files and Line Numbers

Locate the exact line numbers for malformed Unicode sequences in the affected files.

In [5]:
affected_details = []

for path, kind, detail in invalid_files:
    try:
        raw = path.read_bytes()
        text = raw.decode('utf-8', errors='surrogatepass')
    except Exception as e:
        affected_details.append((path, kind, 'failed decode for detail extraction', str(e)))
        continue

    lines = text.splitlines(keepends=True)
    line_hits = []
    for lineno, line in enumerate(lines, start=1):
        for i, ch in enumerate(line):
            if 0xD800 <= ord(ch) <= 0xDFFF:
                line_hits.append((lineno, i, hex(ord(ch)), line.rstrip('\n')))
    if kind == 'utf8-decode-fail':
        line_hits.append(('decode failure', detail, None, None))
    affected_details.append((path, kind, line_hits))

for path, kind, hits in affected_details:
    print(path.relative_to(root), kind)
    for hit in hits[:10]:
        print(' ', hit)

affected_details

[]

## Replace Invalid Characters Safely

Replace malformed Unicode sequences with valid UTF-8 equivalents while preserving surrounding text, formatting, comments, and code behavior.

Invalid surrogate code points should be replaced with a safe placeholder or removed if they are not meaningful. If the file contains invalid UTF-8 bytes, the bytes are decoded with replacement and then rewritten as UTF-8.

In [6]:
def replace_invalid_unicode(text):
    return ''.join(ch if not (0xD800 <= ord(ch) <= 0xDFFF) else '\uFFFD' for ch in text)

fixed_files = []

for path, kind, detail in invalid_files:
    raw = path.read_bytes()
    if kind == 'utf8-decode-fail':
        text = raw.decode('utf-8', errors='replace')
    else:
        text = raw.decode('utf-8', errors='surrogatepass')
    fixed_text = replace_invalid_unicode(text)
    if text != fixed_text:
        path.write_text(fixed_text, encoding='utf-8')
        fixed_files.append((path, kind, 'replaced invalid Unicode'))

fixed_files

[]

## Enforce UTF-8 Encoding on Text Files

Rewrite all likely text files as UTF-8 without BOM to ensure consistent encoding across the project.

In [ ]:
reencoded_files = []

for path in root.rglob('*'):
    if not path.is_file():
        continue
    if path.suffix.lower() in skip_extensions:
        continue
    is_text = path.suffix.lower() in text_extensions or path.name in {'package.json', '.gitignore', 'README.md', 'LICENSE', 'vite.config.js', 'eslint.config.js'}
    if not is_text and path.stat().st_size > 1024 * 1024:
        continue

    data = path.read_bytes()
    try:
        text = data.decode('utf-8')
    except Exception:
        try:
            text = data.decode('utf-8', errors='surrogatepass')
        except Exception:
            continue
    path.write_text(text, encoding='utf-8')
    reencoded_files.append(path)

len(reencoded_files)

## Validate Fixes and Summarize Changes

Re-scan the workspace to confirm no malformed Unicode remains and summarize modified files.

In [7]:
rescan_issues = []

for path in root.rglob('*'):
    if not path.is_file():
        continue
    if path.suffix.lower() in skip_extensions:
        continue
    data = path.read_bytes()
    try:
        text = data.decode('utf-8')
    except Exception as e:
        rescan_issues.append((path, 'utf8-decode-fail', str(e)))
        continue
    surgs = [(i, hex(ord(ch)), text.count('\n', 0, i) + 1) for i, ch in enumerate(text) if 0xD800 <= ord(ch) <= 0xDFFF]
    if surgs:
        rescan_issues.append((path, 'surrogates', surgs[:20]))

print('Rescan issues count:', len(rescan_issues))
for path, kind, detail in rescan_issues:
    print(path.relative_to(root), kind, detail)

print('\nSummary of modified files:')
for path, kind, reason in fixed_files:
    print('-', path.relative_to(root), kind, reason)

Rescan issues count: 413
node_modules\lightningcss-win32-x64-msvc\lightningcss.win32-x64-msvc.node utf8-decode-fail 'utf-8' codec can't decode byte 0x90 in position 2: invalid start byte
node_modules\@rolldown\binding-win32-x64-msvc\rolldown-binding.win32-x64-msvc.node utf8-decode-fail 'utf-8' codec can't decode byte 0x90 in position 2: invalid start byte
.venv\Lib\site-packages\pip\__pycache__\__init__.cpython-313.pyc utf8-decode-fail 'utf-8' codec can't decode byte 0xf3 in position 0: invalid continuation byte
.venv\Lib\site-packages\pip\__pycache__\__main__.cpython-313.pyc utf8-decode-fail 'utf-8' codec can't decode byte 0xf3 in position 0: invalid continuation byte
.venv\Lib\site-packages\pip\__pycache__\__pip-runner__.cpython-313.pyc utf8-decode-fail 'utf-8' codec can't decode byte 0xf3 in position 0: invalid continuation byte
.venv\Lib\site-packages\pip\_vendor\__pycache__\typing_extensions.cpython-313.pyc utf8-decode-fail 'utf-8' codec can't decode byte 0xf3 in position 0: inval

In [9]:
ignore_dirs = {'node_modules', '.venv', 'dist', '__pycache__', '.git'}
project_issues = []
extra_names = {'package.json', '.gitignore', 'README.md', 'LICENSE', 'vite.config.js', 'eslint.config.js'}
for path in root.rglob('*'):
    if not path.is_file():
        continue
    if any(part in ignore_dirs for part in path.parts):
        continue
    if path.suffix.lower() in skip_extensions:
        continue
    if path.suffix.lower() not in text_extensions and path.name not in extra_names:
        continue
    data = path.read_bytes()
    try:
        text = data.decode('utf-8')
    except Exception as e:
        project_issues.append((path, 'utf8-decode-fail', str(e)))
        continue
    for i, ch in enumerate(text):
        if 0xD800 <= ord(ch) <= 0xDFFF:
            project_issues.append((path, 'surrogate', i, hex(ord(ch)), text.count('\n', 0, i) + 1))
            break
print('Project issues count:', len(project_issues))
for issue in project_issues:
    print(issue)


Project issues count: 0
